# LLM 실습 문제 (Practice)

각 셀에서 `### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!` 부분을 채워보세요.

GPT from scratch: Tokenization → Dataset/DataLoader → Embedding → Attention → Transformer → Pretraining → Inference → Fine-tuning (LoRA / Classification / Instruction / DPO)

## Setup

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 문제 1. BPE Tokenizer encode / decode

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)

integers = tokenizer.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(text, allowed_special={"<|endoftext|>"})
print(integers)

strings = tokenizer.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(integers)
print(strings)

## 문제 2. Next-Token Prediction: 입력(x) / 정답(y) 쌍 만들기

In [ ]:
enc_text = tokenizer.encode(raw_text)
enc_sample = enc_text[50:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!:context_size+### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!]

print(f"x: {x}")
print(f"y:      {y}")

## 문제 3. GPTDataset: sliding window로 (input, target) 생성

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "토큰화된 입력의 개수는 적어도 max_length+1과 같아야 합니다."

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! : ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! + max_length]
            target_chunk = token_ids[### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! + ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! : ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! + max_length + ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!], self.target_ids[### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!]

## 문제 4. Token Embedding + Positional Embedding

In [ ]:
vocab_size = 50257
output_dim = 256
max_length = 4

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

pos_embedding_layer = torch.nn.Embedding(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!))
print(pos_embeddings.shape)

input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

## 문제 5. Causal (Masked) Self-Attention

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()

        self.W_query = nn.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, bias=qkv_bias)
        self.W_key   = nn.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, bias=qkv_bias)
        self.W_value = nn.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!)
        queries = self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!)
        values = self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!)

        attn_scores = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! @ ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],
            ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        )
        attn_weights = torch.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

        return context_vec

## 문제 6. Multi-Head Attention (head 분할/결합)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        values = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        queries = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

        attn_scores = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

        mask_bool = self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2)

        context_vec = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        context_vec = self.out_proj(context_vec)

        return context_vec

## 문제 7. LayerNorm

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        var = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        norm_x = (x - ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!) / torch.sqrt(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! + self.eps)
        return self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! * norm_x + self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

## 문제 8. FeedForward (GPT-style MLP)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! * cfg["emb_dim"]),
            ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(),
            nn.Linear(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

## 문제 9. Transformer Block forward (Residual 연결 포함)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x

## 문제 10. GPTModel (전체 아키텍처 조립 + forward)

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!)

        x = tok_embeds + ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!
        x = self.drop_emb(x)
        x = self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(x)
        x = self.final_norm(x)
        logits = self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(x)

        return logits

## 문제 11. Greedy Text Generation Loop

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]

        with torch.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!():
            logits = model(idx_cond)
        logits = logits[:, ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, :]

        idx_next = torch.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(logits, dim=-1, keepdim=True)
        idx = torch.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!((idx, idx_next), dim=1)

    return idx

## 문제 12. Pretraining Loss (Cross Entropy over Vocab)

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(
        logits.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(0, 1),
        target_batch.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!()
    )
    return loss

## 문제 13. PyTorch 학습 루프 (모드 전환 + zero_grad / backward / step)

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!()

        for input_batch, target_batch in train_loader:
            optimizer.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!()

            loss = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(input_batch, target_batch, model, device)

            loss.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!()

            optimizer.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!()

            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

    return train_losses, val_losses, track_tokens_seen

## 문제 14. Text ↔ Token IDs 변환 (batch 차원 추가/제거)

In [ ]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(0)
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(0)
    return tokenizer.decode(flat.tolist())

## 문제 15. LoRA (Low-Rank Adaptation) 레이어

In [ ]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.empty(in_dim, ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

        self.B = nn.Parameter(torch.zeros(### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, out_dim))

        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! / self.rank) * (x @ self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!! @ self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!)
        return x


class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        return self.linear(x) + self.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(x)


def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!):
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            replace_linear_with_lora(module, rank, alpha)

## 문제 16. LoRA 적용 (전체 freeze → LoRA만 학습)

In [ ]:
for param in model.parameters():
    param.requires_grad = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(model, rank=LORA_RANK, alpha=LORA_ALPHA)

model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"총 학습 가능 LoRA 파라미터 수: {total_params:,}")

## 문제 17. 분류 Fine-tuning (마지막 토큰 logit + 선택적 freeze)

In [ ]:
for param in model.parameters():
    param.requires_grad = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

torch.manual_seed(123)
num_classes = 2
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!)

model.to(device)

for param in model.trf_blocks[### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!].parameters():
    param.requires_grad = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!

for param in model.final_norm.parameters():
    param.requires_grad = ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!


def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)[:, ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, :]
    loss = torch.nn.functional.cross_entropy(logits, target_batch)
    return loss

## 문제 18. Instruction Finetuning용 Collate 함수 (Shift + Padding + Mask)

In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)

    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!]

        padded = new_item + [### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!] * (batch_max_length - len(new_item))

        inputs = torch.tensor(padded[:### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!])
        targets = torch.tensor(padded[### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

## 문제 19. DPO Log Probability & Loss

In [ ]:
def compute_logprobs(logits, labels, selection_mask=None):
    labels = labels[:, ### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!:].clone()
    logits = logits[:, :### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!, :]

    log_probs = F.log_softmax(logits, dim=-1)

    selected_log_probs = torch.gather(
        input=### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    if selection_mask is not None:
        mask = selection_mask[:, 1:].clone()
        selected_log_probs = selected_log_probs * mask
        avg_log_prob = selected_log_probs.sum(-1) / mask.sum(-1)
        return avg_log_prob
    else:
        return selected_log_probs.mean(-1)


def compute_dpo_loss(model_chosen_logprobs, model_rejected_logprobs,
                     reference_chosen_logprobs, reference_rejected_logprobs, beta=0.1):
    model_logratios = model_chosen_logprobs - model_rejected_logprobs
    reference_logratios = reference_chosen_logprobs - reference_rejected_logprobs

    logits = model_logratios - reference_logratios

    losses = -F.### !!!!!!!!!!!!코드를 작성하세요!!!!!!!!!!!!(beta * logits)

    chosen_rewards = (model_chosen_logprobs - reference_chosen_logprobs).detach()
    rejected_rewards = (model_rejected_logprobs - reference_rejected_logprobs).detach()

    return losses.mean(), chosen_rewards.mean(), rejected_rewards.mean()